# Peptide-MHC 3D Structure Generation with ColabFold

## Install ColabFold

Run once per Colab session. Takes ~3-5 minutes. Installs ColabFold and its dependencies.

In [ ]:
import os, platform, multiprocessing, shutil

N_CPU = multiprocessing.cpu_count()
IS_MAC = platform.system() == "Darwin"
IS_ARM_MAC = IS_MAC and platform.machine() == "arm64"

TRY_JAX_METAL = False

os.environ["XLA_FLAGS"] = (
    os.environ.get("XLA_FLAGS", "")
    + f" --xla_cpu_multi_thread_eigen=true"
    + f" intra_op_parallelism_threads={N_CPU}"
)
os.environ.setdefault("OMP_NUM_THREADS", str(N_CPU))
os.environ.setdefault("MKL_NUM_THREADS", str(N_CPU))
os.environ.setdefault("OPENBLAS_NUM_THREADS", str(N_CPU))

os.environ.setdefault("XLA_PYTHON_CLIENT_MEM_FRACTION", "0.95")
os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "true")
os.environ.setdefault("TF_FORCE_UNIFIED_MEMORY", "1")

if not os.path.isfile("COLABFOLD_READY"):
    print("Installing ColabFold (one-time, ~3-5 min)...")
    os.system("pip install -q 'colabfold[alphafold-minus-jax] @ git+https://github.com/sokrypton/ColabFold'")

    if IS_MAC:
        os.system("pip install -q 'jax==0.4.35' 'jaxlib==0.4.35'")
        if TRY_JAX_METAL and IS_ARM_MAC:
            print("Attempting experimental jax-metal (Apple GPU)...")
            os.system("pip install -q jax-metal")
    else:
        os.system("pip install -q 'jax[cuda12]==0.4.35'")

    if shutil.which("conda"):
        os.system("conda install -y -c conda-forge openmm pdbfixer")
    else:
        os.system("pip install -q openmm 'pdbfixer @ git+https://github.com/openmm/pdbfixer.git'")

    os.system("touch COLABFOLD_READY")
    print("Done.")
else:
    print("ColabFold already installed.")

import jax
DEVICES = jax.devices()
ACCEL = DEVICES[0].platform        
print(f"JAX {jax.__version__} | backend = {ACCEL.upper()} | devices = {DEVICES} | CPU cores = {N_CPU}")
if ACCEL == "cpu":
    print("NOTE: running on CPU -> maxed across all cores, but folds are slow. "
          "Use a CUDA GPU machine for large batches.")


In [2]:
hla_library = {
    "HLA-A*24:02": ("GSHSMRYFSTSVSRPGRGEPRFIAVGYVDDTQFVRFDSDAASQRMEPRAPWIEQEGPEYW"
        "DEETGKVKAHSQTDRENLRIALRYYNQSEAGSHTLQMMFGCDVGSDGRFLRGYHQYAYDG"
        "KDYIALKEDLRSWTAADMAAQITKRKWEAAHVAEQQRAYLEGTCVDGLRRYLENGKETLQ"
        "RTDPPKTHMTHHPISDHEATLRCWALGFYPAEITLTWQRDGEDQTQDTELVETRPAGDGT"
        "FQKWAAVVVPSGEEQRYTCHVQHEGLPKPLTLRW"),
    "HLA-A*02:01": ("GSHSMRYFFTSVSRPGRGEPRFIAVGYVDDTQFVRFDSDAASQRMEPRAPWIEQEGPEYW"
        "DGETRKVKAHSQTHRVDLGTLRGYYNQSEAGSHTVQRMYGCDVGSDWRFLRGYHQYAYDG"
        "KDYIALKEDLRSWTAADMAAQTTKHKWEAAHVAEQLRAYLEGTCVEWLRRYLENGKETLQ"
        "RTDAPKTHMTHHAVSDHEATLRCWALSFYPAEITLTWQRDGEDQTQDTELVETRPAGDGT"
        "FQKWAAVVVPSGQEQRYTCHVQHEGLPKPLTLRW"),
    "HLA-A*32:01": ("GSHSMRYFFTSVSRPGRGEPRFIAVGYVDDTQFVRFDSDAASQRMEPRAPWIEQEGPEYW"
        "DQETRNVKAHSQTDRESLRIALRYYNQSEAGSHTIQMMYGCDVGPDGRLLRGYQQDAYDG"
        "KDYIALNEDLRSWTAADMAAQITQRKWEAARVAEQLRAYLEGTCVEWLRRYLENGKETLQ"
        "RTDAPKTHMTHHAVSDHEATLRCWALSFYPAEITLTWQRDGEDQTQDTELVETRPAGDGT"
        "FQKWASVVVPSGQEQRYTCHVQHEGLPKPLTLRW"),
    "HLA-B*15:10": ("GSHSMRYFYTAMSRPGRGEPRFISVGYVDDTQFVRFDSDAASPREEPRAPWIEQEGPEYW"
        "DRNTQICKTNTQTYRESLRNLRGYYNQSEAGSHTLQRMYGCDVGPDGRLLRGHDQYAYDG"
        "KDYIALNEDLSSWTAADTAAQITQRKWEAAREAEQLRAYLEGLCVEWLRRYLENGKETLQ"
        "RADPPKTHVTHHPISDHEATLRCWALGFYPAEITLTWQRDGEDQTQDTELVETRPAGDRT"
        "FQKWAAVVVPSGEEQRYTCHVQHEGLPKPLTLRW"),
    "HLA-C*03:04": ("GSHSMRYFYTAVSRPGRGEPHFIAVGYVDDTQFVRFDSDAASPRGEPRAPWVEQEGPEYW"
        "DRETQKYKRQAQTDRVSLRNLRGYYNQSEAGSHIIQRMYGCDVGPDGRLLRGYDQYAYDG"
        "KDYIALNEDLRSWTAADTAAQITQRKWEAAREAEQLRAYLEGLCVEWLRRYLKNGKETLQ"
        "RAEHPKTHVTHHPVSDHEATLRCWALGFYPAEITLTWQWDGEDQTQDTELVETRPAGDGT"
        "FQKWAAVVVPSGEEQRYTCHVQHEGLPEPLTLRW"),
    "HLA-C*14:02": ("CSHSMRYFSTSVSRPGRGEPRFIAVGYVDDTQFVRFDSDAASPRGEPRAPWVEQEGPEYW"
        "DRETQKYKRQAQTDRVSLRNLRGYYNQSEAGSHTLQWMFGCDLGPDGRLLRGYDQSAYDG"
        "KDYIALNEDLRSWTAADTAAQITQRKWEAAREAEQRRAYLEGTCVEWLRRYLENGKETLQ"
        "RAEHPKTHVTHHPVSDHEATLRCWALGFYPAEITLTWQWDGEDQTQDTELVETRPAGDGT"
        "FQKWAAVVVPSGEEQRYTCHVQHEGLPEPLTLRW"),
    "HLA-B*44:02": ("GSHSMRYFYTAMSRPGRGEPRFITVGYVDDTLFVRFDSDATSPRKEPRAPWIEQEGPEYW"
        "DRETQISKTNTQTYRENLRTALRYYNQSEAGSHIIQRMYGCDVGPDGRLLRGYDQDAYDG"
        "KDYIALNEDLSSWTAADTAAQITQRKWEAARVAEQDRAYLEGLCVESLRRYLENGKETLQ"
        "RADPPKTHVTHHPISDHEVTLRCWALGFYPAEITLTWQRDGEDQTQDTELVETRPAGDRT"
        "FQKWAAVVVPSGEEQRYTCHVQHEGLPKPLTLRW"),
    "HLA-B*57:01": ("GSHSMRYFYTAMSRPGRGEPRFIAVGYVDDTQFVRFDSDAASPRMAPRAPWIEQEGPEYW"
        "DGETRNMKASAQTYRENLRIALRYYNQSEAGSHIIQVMYGCDVGPDGRLLRGHDQSAYDG"
        "KDYIALNEDLSSWTAADTAAQITQRKWEAARVAEQLRAYLEGLCVEWLRRYLENGKETLQ"
        "RADPPKTHVTHHPISDHEATLRCWALGFYPAEITLTWQRDGEDQTQDTELVETRPAGDRT"
        "FQKWAAVVVPSGEEQRYTCHVQHEGLPKPLTLRW"),
    "HLA-A*03:01": ("GSHSMRYFFTSVSRPGRGEPRFIAVGYVDDTQFVRFDSDAASQRMEPRAPWIEQEGPEYW"
        "DQETRNVKAQSQTDRVDLGTLRGYYNQSEAGSHTIQIMYGCDVGSDGRFLRGYRQDAYDG"
        "KDYIALNEDLRSWTAADMAAQITKRKWEAAHEAEQLRAYLDGTCVEWLRRYLENGKETLQ"
        "RTDPPKTHMTHHPISDHEATLRCWALGFYPAEITLTWQRDGEDQTQDTELVETRPAGDGT"
        "FQKWAAVVVPSGEEQRYTCHVQHEGLPKPLTLRW"),
    "HLA-C*05:01": ("CSHSMRYFYTAVSRPGRGEPRFIAVGYVDDTQFVQFDSDAASPRGEPRAPWVEQEGPEYW"
        "DRETQKYKRQAQTDRVNLRKLRGYYNQSEAGSHTLQRMYGCDLGPDGRLLRGYNQFAYDG"
        "KDYIALNEDLRSWTAADKAAQITQRKWEAAREAEQRRAYLEGTCVEWLRRYLENGKKTLQ"
        "RAEHPKTHVTHHPVSDHEATLRCWALGFYPAEITLTWQRDGEDQTQDTELVETRPAGDGT"
        "FQKWAAVVVPSGEEQRYTCHVQHEGLPEPLTLRW"),
    "HLA-B*27:05": ("GSHSMRYFHTSVSRPGRGEPRFITVGYVDDTLFVRFDSDAASPREEPRAPWIEQEGPEYW"
        "DRETQICKAKAQTDREDLRTLLRYYNQSEAGSHTLQNMYGCDVGPDGRLLRGYHQDAYDG"
        "KDYIALNEDLSSWTAADTAAQITQRKWEAARVAEQLRAYLEGECVEWLRRYLENGKETLQ"
        "RADPPKTHVTHHPISDHEATLRCWALGFYPAEITLTWQRDGEDQTQDTELVETRPAGDRT"
        "FQKWAAVVVPSGEEQRYTCHVQHEGLPKPLTLRW"),
    "HLA-B*51:01": ("GSHSMRYFYTAMSRPGRGEPRFIAVGYVDDTQFVRFDSDAASPRTEPRAPWIEQEGPEYW"
        "DRNTQIFKTNTQTYRENLRIALRYYNQSEAGSHTWQTMYGCDVGPDGRLLRGHNQYAYDG"
        "KDYIALNEDLSSWTAADTAAQITQRKWEAAREAEQLRAYLEGLCVEWLRRHLENGKETLQ"
        "RADPPKTHVTHHPVSDHEATLRCWALGFYPAEITLTWQRDGEDQTQDTELVETRPAGDRT"
        "FQKWAAVVVPSGEEQRYTCHVQHEGLPKPLTLRW"),
    "HLA-B*40:01": ("GSHSMRYFHTAMSRPGRGEPRFITVGYVDDTLFVRFDSDATSPRKEPRAPWIEQEGPEYW"
        "DRETQISKTNTQTYRESLRNLRGYYNQSEAGSHTLQRMYGCDVGPDGRLLRGHNQYAYDG"
        "KDYIALNEDLRSWTAADTAAQISQRKLEAARVAEQLRAYLEGECVEWLRRYLENGKDKLE"
        "RADPPKTHVTHHPISDHEATLRCWALGFYPAEITLTWQRDGEDQTQDTELVETRPAGDRT"
        "FQKWAAVVVPSGEEQRYTCHVQHEGLPKPLTLRW"),
    "HLA-B*40:02": ("GSHSMRYFHTSVSRPGRGEPRFITVGYVDDTLFVRFDSDATSPRKEPRAPWIEQEGPEYW"
        "DRETQISKTNTQTYRESLRNLRGYYNQSEAGSHTLQSMYGCDVGPDGRLLRGHNQYAYDG"
        "KDYIALNEDLRSWTAADTAAQITQRKWEAARVAEQLRAYLEGECVEWLRRYLENGKETLQ"
        "RADPPKTHVTHHPISDHEATLRCWALGFYPAEITLTWQRDGEDQTQDTELVETRPAGDRT"
        "FQKWAAVVVPSGEEQRYTCHVQHEGLPKPLTLRW"),
    "HLA-B*07:02": ("GSHSMRYFYTSVSRPGRGEPRFISVGYVDDTQFVRFDSDAASPREEPRAPWIEQEGPEYW"
        "DRNTQIYKAQAQTDRESLRNLRGYYNQSEAGSHTLQSMYGCDVGPDGRLLRGHDQYAYDG"
        "KDYIALNEDLRSWTAADTAAQITQRKWEAAREAEQRRAYLEGECVEWLRRYLENGKDKLE"
        "RADPPKTHVTHHPISDHEATLRCWALGFYPAEITLTWQRDGEDQTQDTELVETRPAGDRT"
        "FQKWAAVVVPSGEEQRYTCHVQHEGLPKPLTLRW"),
    "HLA-B*18:01": ("GSHSMRYFHTSVSRPGRGEPRFISVGYVDGTQFVRFDSDAASPRTEPRAPWIEQEGPEYW"
        "DRNTQISKTNTQTYRESLRNLRGYYNQSEAGSHTLQRMYGCDVGPDGRLLRGHDQSAYDG"
        "KDYIALNEDLSSWTAADTAAQITQRKWEAARVAEQLRAYLEGTCVEWLRRHLENGKETLQ"
        "RADPPKTHVTHHPISDHEATLRCWALGFYPAEITLTWQRDGEDQTQDTELVETRPAGDRT"
        "FQKWAAVVVPSGEEQRYTCHVQHEGLPKPLTLRW"),
    "HLA-A*25:01": ("GSHSMRYFYTSVSRPGRGEPRFIAVGYVDDTQFVRFDSDAASQRMEPRAPWIEQEGPEYW"
        "DRNTRNVKAHSQTDRESLRIALRYYNQSEDGSHTIQRMYGCDVGPDGRFLRGYQQDAYDG"
        "KDYIALNEDLRSWTAADMAAQITQRKWETAHEAEQWRAYLEGRCVEWLRRYLENGKETLQ"
        "RTDAPKTHMTHHAVSDHEATLRCWALSFYPAEITLTWQRDGEDQTQDTELVETRPAGDGT"
        "FQKWASVVVPSGQEQRYTCHVQHEGLPKPLTLRW"),
    "HLA-B*44:03": ("GSHSMRYFYTAMSRPGRGEPRFITVGYVDDTLFVRFDSDATSPRKEPRAPWIEQEGPEYW"
        "DRETQISKTNTQTYRENLRTALRYYNQSEAGSHIIQRMYGCDVGPDGRLLRGYDQDAYDG"
        "KDYIALNEDLSSWTAADTAAQITQRKWEAARVAEQLRAYLEGLCVESLRRYLENGKETLQ"
        "RADPPKTHVTHHPISDHEVTLRCWALGFYPAEITLTWQRDGEDQTQDTELVETRPAGDRT"
        "FQKWAAVVVPSGEEQRYTCHVQHEGLPKPLTLRW"),
    "HLA-A*30:01": ("GSHSMRYFSTSVSRPGSGEPRFIAVGYVDDTQFVRFDSDAASQRMEPRAPWIEQERPEYW"
        "DQETRNVKAQSQTDRVDLGTLRGYYNQSEAGSHTIQIMYGCDVGSDGRFLRGYEQHAYDG"
        "KDYIALNEDLRSWTAADMAAQITQRKWEAARWAEQLRAYLEGTCVEWLRRYLENGKETLQ"
        "RTDPPKTHMTHHPISDHEATLRCWALGFYPAEITLTWQRDGEDQTQDTELVETRPAGDGT"
        "FQKWAAVVVPSGEEQRYTCHVQHEGLPKPLTLRW"),
    "HLA-A*11:01": ("GSHSMRYFYTSVSRPGRGEPRFIAVGYVDDTQFVRFDSDAASQRMEPRAPWIEQEGPEYW"
        "DQETRNVKAQSQTDRVDLGTLRGYYNQSEDGSHTIQIMYGCDVGPDGRFLRGYRQDAYDG"
        "KDYIALNEDLRSWTAADMAAQITKRKWEAAHAAEQQRAYLEGRCVEWLRRYLENGKETLQ"
        "RTDPPKTHMTHHPISDHEATLRCWALGFYPAEITLTWQRDGEDQTQDTELVETRPAGDGT"
        "FQKWAAVVVPSGEEQRYTCHVQHEGLPKPLTLRW"),
    "HLA-C*12:03": ("CSHSMRYFYTAVSRPGRGEPRFIAVGYVDDTQFVRFDSDAASPRGEPRAPWVEQEGPEYW"
        "DRETQKYKRQAQADRVSLRNLRGYYNQSEAGSHTLQWMYGCDLGPDGRLLRGYDQSAYDG"
        "KDYIALNEDLRSWTAADTAAQITQRKWEAAREAEQWRAYLEGTCVEWLRRYLENGKETLQ"
        "RAEHPKTHVTHHPVSDHEATLRCWALGFYPAEITLTWQRDGEDQTQDTELVETRPAGDGT"
        "FQKWAAVVVPSGEEQRYTCHVQHEGLPEPLTLRW"),
    "HLA-B*49:01": ("GSHSMRYFHTAMSRPGRGEPRFITVGYVDDTLFVRFDSDATSPRKEPRAPWIEQEGPEYW"
        "DRETQISKTNTQTYRENLRIALRYYNQSEAGSHTWQRMYGCDLGPDGRLLRGYNQLAYDG"
        "KDYIALNEDLSSWTAADTAAQITQRKWEAAREAEQLRAYLEGLCVEWLRRYLENGKETLQ"
        "RADPPKTHVTHHPISDHEATLRCWALGFYPAEITLTWQRDGEDQTQDTELVETRPAGDRT"
        "FQKWAAVVVPSGEEQRYTCHVQHEGLPKPLTLRW"),
    "HLA-A*01:01": ("GSHSMRYFFTSVSRPGRGEPRFIAVGYVDDTQFVRFDSDAASQKMEPRAPWIEQEGPEYW"
        "DQETRNMKAHSQTDRANLGTLRGYYNQSEDGSHTIQIMYGCDVGPDGRFLRGYRQDAYDG"
        "KDYIALNEDLRSWTAADMAAQITKRKWEAVHAAEQRRVYLEGRCVDGLRRYLENGKETLQ"
        "RTDPPKTHMTHHPISDHEATLRCWALGFYPAEITLTWQRDGEDQTQDTELVETRPAGDGT"
        "FQKWAAVVVPSGEEQRYTCHVQHEGLPKPLTLRW"),
    "HLA-B*15:02": ("GSHSMRYFYTAMSRPGRGEPRFIAVGYVDDTQFVRFDSDAASPRMAPRAPWIEQEGPEYW"
        "DRNTQISKTNTQTYRESLRNLRGYYNQSEAGSHIIQRMYGCDVGPDGRLLRGYDQSAYDG"
        "KDYIALNEDLSSWTAADTAAQITQRKWEAAREAEQLRAYLEGLCVEWLRRYLENGKETLQ"
        "RADPPKTHVTHHPISDHEATLRCWALGFYPAEITLTWQRDGEDQTQDTELVETRPAGDRT"
        "FQKWAAVVVPSGEEQRYTCHVQHEGLPKPLTLRW"),
    "HLA-B*35:01": ("GSHSMRYFYTAMSRPGRGEPRFIAVGYVDDTQFVRFDSDAASPRTEPRAPWIEQEGPEYW"
        "DRNTQIFKTNTQTYRESLRNLRGYYNQSEAGSHIIQRMYGCDLGPDGRLLRGHDQSAYDG"
        "KDYIALNEDLSSWTAADTAAQITQRKWEAARVAEQLRAYLEGLCVEWLRRYLENGKETLQ"
        "RADPPKTHVTHHPVSDHEATLRCWALGFYPAEITLTWQRDGEDQTQDTELVETRPAGDRT"
        "FQKWAAVVVPSGEEQRYTCHVQHEGLPKPLTLRW"),
    "HLA-A*68:02": ("GSHSMRYFYTSMSRPGRGEPRFIAVGYVDDTQFVRFDSDAASQRMEPRAPWIEQEGPEYW"
        "DRNTRNVKAQSQTDRVDLGTLRGYYNQSEAGSHTIQRMYGCDVGPDGRFLRGYHQYAYDG"
        "KDYIALKEDLRSWTAADMAAQTTKHKWEAAHVAEQWRAYLEGTCVEWLRRYLENGKETLQ"
        "RTDAPKTHMTHHAVSDHEATLRCWALSFYPAEITLTWQRDGEDQTQDTELVETRPAGDGT"
        "FQKWVAVVVPSGQEQRYTCHVQHEGLPKPLTLRW"),
    "HLA-B*15:01": ("GSHSMRYFYTAMSRPGRGEPRFIAVGYVDDTQFVRFDSDAASPRMAPRAPWIEQEGPEYW"
        "DRETQISKTNTQTYRESLRNLRGYYNQSEAGSHTLQRMYGCDVGPDGRLLRGHDQSAYDG"
        "KDYIALNEDLSSWTAADTAAQITQRKWEAAREAEQWRAYLEGLCVEWLRRYLENGKETLQ"
        "RADPPKTHVTHHPISDHEATLRCWALGFYPAEITLTWQRDGEDQTQDTELVETRPAGDRT"
        "FQKWAAVVVPSGEEQRYTCHVQHEGLPKPLTLRW"),
}

print(f'HLA library alleles: {len(hla_library)}')

HLA library alleles: 27


## Data

In [3]:
import pandas as pd

# from google.colab import drive
# drive.mount('/content/drive')

DATA_PATH = '/Users/shubhaychoubey/Documents/GitHub/PeptideMAIN/Data/ML_Ready/IEDB_ML_ready_with_seq_with_HLA.csv'
OUTPUT_DIR = '/Users/shubhaychoubey/Documents/GitHub/PeptideMAIN/Structures'

df = pd.read_csv(DATA_PATH)
print(f'Loaded {len(df):,} rows')

# Identify the HLA column — adjust name if different in your file
HLA_COL = 'Best HLA Allele' if 'Best HLA Allele' in df.columns else 'allele'
PEP_COL = 'Peptide Sequence'

# Keep only rows whose allele is in our library
df_valid = df[df[HLA_COL].isin(hla_library.keys())].copy()
print(f'Rows with a known allele: {len(df_valid):,}')

# Deduplicate to unique peptide-allele pairs
unique_pairs = df_valid[[PEP_COL, HLA_COL]].drop_duplicates().reset_index(drop=True)
print(f'Unique peptide-allele pairs to fold: {len(unique_pairs):,}')
print(f'(This is what determines total fold time, not the full row count.)')

Loaded 51,788 rows
Rows with a known allele: 43,678
Unique peptide-allele pairs to fold: 37,711
(This is what determines total fold time, not the full row count.)


In [4]:
print(f"Total: {len(df):,}")
print(f"Kept (allele in library): {len(df_valid):,}")
print(f"Dropped: {len(df) - len(df_valid):,}")
print("Alleles not in library:")
print(df[~df[HLA_COL].isin(hla_library.keys())][HLA_COL].value_counts())

Total: 51,788
Kept (allele in library): 43,678
Dropped: 8,110
Alleles not in library:
Best HLA Allele
HLA-B*08:01    569
HLA-B*14:02    559
HLA-C*12:02    537
HLA-A*68:01    482
HLA-A*29:02    396
              ... 
HLA-B*42:01      3
HLA-B*15:11      2
HLA-A*02:06      2
HLA-B*35:08      1
HLA-B*27:07      1
Name: count, Length: 62, dtype: int64


## Folding 

In [ ]:
import os
from pathlib import Path

os.makedirs(OUTPUT_DIR, exist_ok=True)

import jax
from colabfold.batch import run
from colabfold.download import default_data_dir

ACCEL = jax.devices()[0].platform
ON_GPU = (ACCEL == "gpu")          # CUDA only

# ---------------------------------------------------------------------------
# "Max power" generation settings.
#   - All 5 AF2-multimer models, ranked.
#   - Recycles with early-stop: high cap, but bail once the structure
#     converges (best quality without wasting compute).
#   - Amber relaxation of the top model (use GPU relax only on CUDA).
# On CPU these settings are accurate but SLOW; drop NUM_MODELS / NUM_RECYCLES
# if you just want a quick local sanity batch.
# ---------------------------------------------------------------------------
NUM_MODELS              = 5
NUM_RECYCLES            = 6 if ON_GPU else 3     # cap; early-stop usually ends sooner
RECYCLE_EARLY_STOP_TOL  = 0.5                    # stop when pLDDT/pae plateaus
NUM_SEEDS               = 1

try:
    import openmm, pdbfixer  
    NUM_RELAX = 1                                
except ImportError:
    NUM_RELAX = 0
    print('openmm/pdbfixer not found -> skipping Amber relaxation. '
          'Install with: conda install -c conda-forge openmm pdbfixer')

USE_GPU_RELAX           = ON_GPU
MSA_MODE                = "single_sequence"      

print(f"Generation config -> backend={ACCEL.upper()} models={NUM_MODELS} "
      f"recycles<= {NUM_RECYCLES} relax={NUM_RELAX} (gpu_relax={USE_GPU_RELAX}) msa={MSA_MODE}")


def safe_name(peptide, allele):
    allele_clean = allele.replace('*', '').replace(':', '').replace('HLA-', '')
    return f'{peptide}_{allele_clean}'

def fold_pair(peptide, allele, output_dir):
    name = safe_name(peptide, allele)
    result_subdir = os.path.join(output_dir, name)

    if os.path.isdir(result_subdir):
        existing = [f for f in os.listdir(result_subdir) if f.endswith('.pdb') and 'rank_001' in f]
        if existing:
            return os.path.join(result_subdir, existing[0])

    os.makedirs(result_subdir, exist_ok=True)
    mhc_seq = hla_library[allele]

    # When calling run() directly, a complex must be a LIST of chain
    # sequences. The ':' separator is only parsed when ColabFold reads
    # from CSV/FASTA via get_queries(); passing "PEP:MHC" as one string
    # makes it a monomer and the ':' triggers "Invalid character in the
    # sequence: :". So split the two chains here.
    chain_seqs = [peptide, mhc_seq]

    # Keep a human-readable colon-joined copy in the CSV for reference.
    query_csv = os.path.join(result_subdir, 'query.csv')
    with open(query_csv, 'w') as f:
        f.write('id,sequence\n')
        f.write(f"{name},{':'.join(chain_seqs)}\n")

    try:
        # This ColabFold version unpacks each query as a 4-tuple:
        # (jobname, sequence, a3m_lines, template_features)
        run(
            queries=[(name, chain_seqs, None, None)],
            result_dir=result_subdir,
            is_complex=True,
            msa_mode=MSA_MODE,
            num_models=NUM_MODELS,
            num_recycles=NUM_RECYCLES,
            recycle_early_stop_tolerance=RECYCLE_EARLY_STOP_TOL,
            num_seeds=NUM_SEEDS,
            num_relax=NUM_RELAX,
            use_gpu_relax=USE_GPU_RELAX,
            model_type='alphafold2_multimer_v3',
            data_dir=default_data_dir,
            keep_existing_results=True,
            rank_by='auto',
            pair_mode='unpaired',
            stop_at_score=100,
            zip_results=False,
        )
        # Prefer the relaxed top model if present, else the unrelaxed one.
        ranked = sorted(
            f for f in os.listdir(result_subdir)
            if f.endswith('.pdb') and 'rank_001' in f
        )
        if not ranked:
            return None
        relaxed = [f for f in ranked if 'relaxed' in f and 'unrelaxed' not in f]
        best = relaxed[0] if relaxed else ranked[0]
        return os.path.join(result_subdir, best)
    except Exception as e:
        print(f'  FAILED {name}: {e}')
        return None

print('Folding function ready.')

In [ ]:
# Download the AlphaFold2 weights once (~3.5 GB for the multimer set).
# Without this, run() fails with:
#   No such file or directory: .../params/params_model_3_multimer_v3.npz
from colabfold.download import download_alphafold_params, default_data_dir

# Must match model_type used in fold_pair ('alphafold2_multimer_v3').
download_alphafold_params('alphafold2_multimer_v3', default_data_dir)

print('AlphaFold2 multimer weights ready at:', default_data_dir)


AlphaFold2 multimer weights ready at: /Users/shubhaychoubey/Library/Caches/colabfold


In [7]:
import inspect
from colabfold.batch import run
print(inspect.signature(run))

(queries: 'List[Tuple[str, Union[str, List[str]], Optional[List[str]], Optional[List[Tuple[str, str, int]]]]]', result_dir: 'Union[str, Path]', num_models: 'int', is_complex: 'bool', num_recycles: 'Optional[int]' = None, recycle_early_stop_tolerance: 'Optional[float]' = None, model_order: 'List[int]' = [1, 2, 3, 4, 5], initial_guess: 'str' = None, num_ensemble: 'int' = 1, model_type: 'str' = 'auto', msa_mode: 'str' = 'mmseqs2_uniref_env', use_templates: 'bool' = False, custom_template_path: 'str' = None, custom_template_cache_path: 'str' = None, num_relax: 'int' = 0, relax_max_iterations: 'int' = 0, relax_tolerance: 'float' = 2.39, relax_stiffness: 'float' = 10.0, relax_max_outer_iterations: 'int' = 3, keep_existing_results: 'bool' = True, rank_by: 'str' = 'auto', pair_mode: 'str' = 'unpaired_paired', pairing_strategy: 'str' = 'greedy', data_dir: 'Union[str, Path]' = PosixPath('/Users/shubhaychoubey/Library/Caches/colabfold'), host_url: 'str' = 'https://api.colabfold.com', user_agent: 

In [ ]:
import time
from tqdm.notebook import tqdm

manifest_path = os.path.join(OUTPUT_DIR, 'manifest.csv')
manifest = []

start_time = time.time()
n_done, n_failed = 0, 0

for i, row in tqdm(unique_pairs.iterrows(), total=len(unique_pairs)):
    peptide = row[PEP_COL]
    allele  = row[HLA_COL]
    pdb_path = fold_pair(peptide, allele, OUTPUT_DIR)

    if pdb_path:
        n_done += 1
        manifest.append({'peptide': peptide, 'allele': allele, 'pdb_path': pdb_path})
    else:
        n_failed += 1

    if (i + 1) % 50 == 0:
        pd.DataFrame(manifest).to_csv(manifest_path, index=False)
        elapsed = time.time() - start_time
        rate = (i + 1) / elapsed
        remaining = (len(unique_pairs) - i - 1) / rate / 60
        print(f'  {i+1}/{len(unique_pairs)}  done={n_done} failed={n_failed}  '
              f'~{remaining:.0f} min remaining')

pd.DataFrame(manifest).to_csv(manifest_path, index=False)
print(f'\nComplete. {n_done} folded, {n_failed} failed.')
print(f'Manifest saved to {manifest_path}')

  0%|          | 0/37711 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
import numpy as np
from Bio.PDB import PDBParser
from Bio.PDB.SASA import ShrakeRupley

MAX_PEPTIDE_LEN = 15
parser = PDBParser(QUIET=True)
sr = ShrakeRupley()

def extract_features(pdb_path, peptide_len):

    structure = parser.get_structure('cplx', pdb_path)
    model = next(iter(structure))
    chains = list(model)
    peptide_chain = chains[0]  

    ca_coords, plddts = [], []
    for residue in peptide_chain:
        if 'CA' in residue:
            ca_coords.append(residue['CA'].get_coord())
            plddts.append(residue['CA'].get_bfactor())  

    ca_coords = np.array(ca_coords[:peptide_len], dtype=np.float32)
    L = len(ca_coords)

    coords = np.zeros((MAX_PEPTIDE_LEN, 3), dtype=np.float32)
    mask   = np.zeros((MAX_PEPTIDE_LEN,), dtype=bool)
    if L > 0:
        coords[:L] = ca_coords
        mask[:L]   = True

    avg_plddt = float(np.mean(plddts)) if plddts else 0.0

    if L > 1:
        centroid = ca_coords.mean(axis=0)
        rg = float(np.sqrt(np.mean(np.sum((ca_coords - centroid)**2, axis=1))))
    else:
        rg = 0.0

    try:
        sr.compute(peptide_chain, level='C')
        sasa = float(peptide_chain.sasa)
    except Exception:
        sasa = 0.0

    return coords, mask, avg_plddt, rg, sasa

manifest_df = pd.read_csv(manifest_path)
N = len(manifest_df)

all_coords = np.zeros((N, MAX_PEPTIDE_LEN, 3), dtype=np.float32)
all_mask   = np.zeros((N, MAX_PEPTIDE_LEN),    dtype=bool)
feat_rows  = []

for i, row in tqdm(manifest_df.iterrows(), total=N):
    coords, mask, plddt, rg, sasa = extract_features(row['pdb_path'], len(row['peptide']))
    all_coords[i] = coords
    all_mask[i]   = mask
    feat_rows.append({
        'peptide': row['peptide'], 'allele': row['allele'],
        'struct_plddt': plddt, 'struct_radius_gyration': rg, 'struct_sasa': sasa,
    })

np.save(os.path.join(OUTPUT_DIR, 'ca_coords.npy'), all_coords)
np.save(os.path.join(OUTPUT_DIR, 'ca_mask.npy'), all_mask)
struct_features = pd.DataFrame(feat_rows)
struct_features.to_csv(os.path.join(OUTPUT_DIR, 'structure_features.csv'), index=False)

print(f'Saved:')
print(f'  ca_coords.npy        : {all_coords.shape}')
print(f'  ca_mask.npy          : {all_mask.shape}')
print(f'  structure_features.csv: {len(struct_features)} rows')
print(f'\nMean pLDDT: {struct_features.struct_plddt.mean():.1f} (>70 is confident)')

In [ ]:
merged = df_valid.merge(
    struct_features,
    left_on=[PEP_COL, HLA_COL],
    right_on=['peptide', 'allele'],
    how='left'
)
merged = merged.drop(columns=['peptide', 'allele'])

out_path = os.path.join(OUTPUT_DIR, 'mhc_class_I_with_structure.csv')
merged.to_csv(out_path, index=False)

n_with_struct = merged['struct_plddt'].notna().sum()
print(f'Merged dataset: {len(merged):,} rows')
print(f'  rows with structure features: {n_with_struct:,} ({100*n_with_struct/len(merged):.1f}%)')
print(f'  saved to {out_path}')
merged[[PEP_COL, HLA_COL, 'struct_plddt', 'struct_radius_gyration', 'struct_sasa']].head()